<a href="https://colab.research.google.com/github/alvaroanussa13-glitch/Projecto-de-aprendizado-de-m-quina-inibidores-InhA/blob/main/2_Pr%C3%A9_processamento_de_dados_%26_C%C3%A1lculo_de_Features_de_Lipinski_%26_PubChemFP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Desenvolvimento de modelos de machine learning baseados em QSAR-2D para a predição de novos candidatos a fármacos TKI-HER2 para o tratamento de câncer da mama***

---



*@Alvaro Anussa*

### ***2. Machine Learning & QSAR-2D***

### ***2.1. Pré-processamento de dados***

### **Pipeline**
1. Configuração do ambiente de trabalho
2. Carregamento dos dados em .csv
3. Conversão de SMILES em Mol
4. Remoção de sais
5. Padronização de tautômeros
6. Remoção de moléculas não Kekulizadas
7. Remoção de duplicatas
8. Conversão de IC50 em pIC50
9. Binarização de pIC50 em activo/Inactivo

### ***1. Configuração do ambiente de trabalho***

In [1]:
#Instalação de frameworks necessários
!pip install rdkit #Manipulação de dados de compostos químicos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 46.3 MB/s eta 0:00:00


In [2]:
#Instalação de frameworks necessários
import pandas as pd # Tabulação de dados
import numpy as np # Operações matemáticas
from rdkit import Chem # Manipulação e curadoria de dados de compostos químicos
from rdkit.Chem import SaltRemover, MolStandardize # Remoção de sais dos compostos
from rdkit.Chem.MolStandardize import rdMolStandardize # Padronização dos compostos (normalização de tautômeros)
from rdkit.Chem import Descriptors, Lipinski, AllChem # Calcular Features de Lipinski
from rdkit.Chem import rdMolDescriptors # Calcular Fingerprints de PubChem

### ***2. Carregamento dos dados em .csv***

In [4]:
df = pd.read_csv("/content/Dataset_final.csv", sep=";")  # Deve ter colunas 'SMILES' e 'IC50' para pré-processar os compostos

### ***3. Conversão de SMILES em Mol***

In [5]:
df['Mol'] = df['SMILES'].apply(lambda x: Chem.MolFromSmiles(x))
df = df[df['Mol'].notnull()].reset_index(drop=True) # Permite converter os SMILES em arquivo manipulável sob a dimensão química

### ***4. Remoção de sais***

In [6]:
remover = SaltRemover.SaltRemover()
def remover_sais(mol): # Estrututa de uma função que permite remover sais em todos os compostos
    try:
        mol = remover.StripMol(mol, dontRemoveEverything=True)
        return mol
    except:
        return None
df['Mol'] = df['Mol'].apply(remover_sais) # Aplicação da função de remoção de sais
df = df[df['Mol'].notnull()].reset_index(drop=True)

### ***5. Padronização de tautômeros***

In [7]:
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
df['Mol'] = df['Mol'].apply(lambda mol: tautomer_enumerator.Canonicalize(mol))
# Trata conjunto de compostos isómeros transformando em único composto, para evitar duplicação artificial

[13:51:39] Tautomer enumeration stopped at 221 tautomers: max transforms reached
[13:51:41] Tautomer enumeration stopped at 380 tautomers: max transforms reached
[13:51:44] Tautomer enumeration stopped at 186 tautomers: max transforms reached


### ***6. Remoção de moléculas não Kekulizadas***

In [8]:
def mol_valido(mol):
    try:
        Chem.Kekulize(mol, clearAromaticFlags=True)
        return True
    except:
        return False

df['Kekuliza'] = df['Mol'].apply(mol_valido)
df = df[df['Kekuliza']].reset_index(drop=True)
df.drop(columns=['Kekuliza'], inplace=True)
# Remove todos compostos com o sistema aromático anômalo e moléculas com a estrutura central incompleta

### ***7. Remoção de duplicatas***

In [9]:
df['Canonical_SMILES'] = df['Mol'].apply(lambda mol: Chem.MolToSmiles(mol))
df = df.drop_duplicates(subset='Canonical_SMILES').reset_index(drop=True)
# Remove duplicatas geradas após a remoção de sais

### ***8. Conversão de IC50 em pIC50***

In [10]:
# IC50 deve estar em nM (nanomolar)
def converter_para_pIC50(ic50_nM):
    try:
        if ic50_nM > 0:
            return -np.log10(ic50_nM * 1e-9)
        else:
            return np.nan
    except:
        return np.nan

df['pIC50'] = df['IC50_nM'].apply(converter_para_pIC50)
df = df[df['pIC50'].notnull()].reset_index(drop=True)
# Permite converter o IC50 em escala logarítmica para garantir o controle de múltiplas escalas de IC50 em uma logarítmica

### ***9. Categorização de valores de pIC50***

In [11]:
# Definir limiar (pIC50 >= 6.0 → ativo (1); senão → inativo (0))
limiar = 6.0

# Binarizar em 0 e 1
df['Classe'] = df['pIC50'].apply(lambda x: 1 if x >= limiar else 0)

### ***10. Dados pré-processados***

In [12]:
df['Name'] = df.index.astype(str)
df_final = df[['Canonical_SMILES', 'pIC50', 'Classe']]
display(df_final.head())

,Canonical_SMILES,pIC50,Classe
0,O=C(NC1=CC=CC=C1)C1CC(=O)N(C2CCCCC2)C1,4.972243,0
1,O=C(NC1=CC=CC=C1Br)C1CC(=O)N(C2CCCCC2)C1,4.000000,0
2,O=C(NC1=CC2=C(C=C1)OCCO2)C1CC(=O)N(C2CCCCC2)C1,4.000000,0
3,CC1=CC=CC(C)=C1NC(=O)C1CC(=O)N(C2CCCCC2)C1,4.000000,0
4,O=C(NC1=CC=C(OC2=CC=CC=C2)C=C1)C1CC(=O)N(C2CCC...,4.000000,0


### ***2.2. Cálculo de descritores de Lipinski e PubChem Fingerprints***

## **Pipeline**
1. Seleccionar descritores de Lipinski e Fingerprints de PubChem
2. Calcular descritores de Lipinski e PubChemFP para todos SMILES
3. Concatenar com dados pré-processados
4. Salvar os dados para EDA

### ***1. Seleccionar descritores de Lipinski e Fingerprints de PubChem***

In [13]:
def calcular_descritores_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)

    # === Descritores de Lipinski ===
    mol_logp = Descriptors.MolLogP(mol)
    mol_wt = Descriptors.MolWt(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    # === Fingerprint tipo PubChem ===
    fp = Chem.RDKFingerprint(mol, fpSize=881)
    fp_bits = list(fp)

    return [mol_logp, mol_wt, hbd, hba] + fp_bits
# Esta função permite executar os cálculos de features de Lipinski e PubChem Fingerprints

### ***2. Calcular descritores de Lipinski e PubChemFP para todos SMILES***

In [14]:
# Nomes das colunas
colunas_descritores = ['MolLogP', 'MolWt', 'nHBDon', 'nHBAcc']
colunas_fingerprints = [f'PubChemFP_{i}' for i in range(881)]

# Calcular para todos os SMILES
dados = df['SMILES'].apply(calcular_descritores_fp)

df_descritores_fp = pd.DataFrame(dados.tolist(), columns=colunas_descritores + colunas_fingerprints)
# Executa o cálculo das features e insere os resultados na DataFrame

### ***3. Concatenar com dados pré-processados***

In [15]:
# Juntar com informações originais
df_final = pd.concat([df.reset_index(drop=True), df_descritores_fp.reset_index(drop=True)], axis=1)

# Ver resultado
print(df_final.shape)
df_final.head()

(398, 893)


,ChEMBL_ID,SMILES,IC50_nM,Mol,Canonical_SMILES,pIC50,Classe,Name,MolLogP,MolWt,...,PubChemFP_871,PubChemFP_872,PubChemFP_873,PubChemFP_874,PubChemFP_875,PubChemFP_876,PubChemFP_877,PubChemFP_878,PubChemFP_879,PubChemFP_880
0,CHEMBL217926,O=C(Nc1ccccc1)C1CC(=O)N(C2CCCCC2)C1,10660.0,<rdkit.Chem.rdchem.Mol object at 0x7b140eb521f0>,O=C(NC1=CC=CC=C1)C1CC(=O)N(C2CCCCC2)C1,4.972243,0,0,2.80630,286.375,...,0,1,1,1,0,1,1,0,0,0
1,CHEMBL216547,O=C(Nc1ccccc1Br)C1CC(=O)N(C2CCCCC2)C1,100000.0,<rdkit.Chem.rdchem.Mol object at 0x7b140eb522d0>,O=C(NC1=CC=CC=C1Br)C1CC(=O)N(C2CCCCC2)C1,4.000000,0,1,3.56880,365.271,...,0,1,1,1,0,1,1,1,0,0
2,CHEMBL213720,O=C(Nc1ccc2c(c1)OCCO2)C1CC(=O)N(C2CCCCC2)C1,100000.0,<rdkit.Chem.rdchem.Mol object at 0x7b140eb52340>,O=C(NC1=CC2=C(C=C1)OCCO2)C1CC(=O)N(C2CCCCC2)C1,4.000000,0,2,2.57750,344.411,...,1,1,1,1,0,1,1,0,0,0
3,CHEMBL217274,Cc1cccc(C)c1NC(=O)C1CC(=O)N(C2CCCCC2)C1,100000.0,<rdkit.Chem.rdchem.Mol object at 0x7b140eb523b0>,CC1=CC=CC(C)=C1NC(=O)C1CC(=O)N(C2CCCCC2)C1,4.000000,0,3,3.42314,314.429,...,0,1,1,1,0,1,1,0,0,0
4,CHEMBL217773,O=C(Nc1ccc(Oc2ccccc2)cc1)C1CC(=O)N(C2CCCCC2)C1,100000.0,<rdkit.Chem.rdchem.Mol object at 0x7b140eb52420>,O=C(NC1=CC=C(OC2=CC=CC=C2)C=C1)C1CC(=O)N(C2CCC...,4.000000,0,4,4.59860,378.472,...,0,1,1,1,0,1,1,0,0,0


### ***4. Salvar os dados para EDA***

In [16]:
df_final.to_csv('Dados para EDA.csv', index=False, encoding='utf-8')

### ***FIM***